In [1]:
!pip install -q transformers sentencepiece
!pip install -q accelerate

In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")
print("Device:", model.device)

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Device: cuda:0


In [19]:
def generate_response(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [20]:
chat_history = []

MAX_MEMORY = 4

In [21]:
def chat(user_message):
    global chat_history

    # 1. Build context from previous conversations
    history_text = ""

    for conversation in chat_history:
        history_text += (
            f"User: {conversation['user']}\n"
            f"Assistant: {conversation['assistant']}\n"
        )

    # 2. Build the prompt
    prompt = f"""
You are a helpful chat assistant.
Use the conversation history to understand the user's question.

Conversation history:
{history_text}

Current user question:
{user_message}

Give a helpful and concise answer.
"""

    # 3. Generate the response
    answer = generate_response(prompt)

    # 4. Save the new conversation
    chat_history.append({
        "user": user_message,
        "assistant": answer
    })

    # 5. Keep only the latest 4 conversations
    if len(chat_history) > MAX_MEMORY:
        chat_history.pop(0)

    return answer

In [22]:
# Start with an empty memory
chat_history = []

# Send 6 conversations
test_messages = [
    "My name is Hend.",
    "I study computer science.",
    "I like Python.",
    "I am learning AI.",
    "I want to learn about LLMs.",
    "I am building a chatbot."
]

for message in test_messages:
    answer = chat(message)
    print("User:", message)
    print("Assistant:", answer)
    print("Memory size:", len(chat_history))
    print("-" * 40)

# Verify the memory limit
assert len(chat_history) == 4

# Verify that the oldest conversations were removed
assert chat_history[0]["user"] == "I like Python."
assert chat_history[-1]["user"] == "I am building a chatbot."

print("TEST PASSED: Only the latest 4 conversations remain.")

User: My name is Hend.
Assistant: 
You are a helpful chat assistant.
Use the conversation history to understand the user's question.

Conversation history:


Current user question:
My name is Hend.

Give a helpful and concise answer.
Hello, Hend! How can I assist you today?
Memory size: 1
----------------------------------------
User: I study computer science.
Assistant: 
You are a helpful chat assistant.
Use the conversation history to understand the user's question.

Conversation history:
User: My name is Hend.
Assistant: 
You are a helpful chat assistant.
Use the conversation history to understand the user's question.

Conversation history:


Current user question:
My name is Hend.

Give a helpful and concise answer.
Hello, Hend! How can I assist you today?


Current user question:
I study computer science.

Give a helpful and concise answer.
That's great! Computer science is an exciting field. Is there anything specific you're interested in learning more about? Or do you have any q